# Build and check a ReLU MILP
# ReLU MILP를 만들고 대조하기

At fixed T=330 K and feed=1.2 mol/L, choose residence time between 2 and 10 min. Require B≥0.40 and C≤0.25 mol/L and nonnegative outputs. With flow 1 L/min, minimizing residence time also minimizes reactor volume. Fit a piecewise-linear surrogate, then reproduce its exact ReLU mapping inside an MILP.

T=330 K, feed=1.2 mol/L에서 2~10분의 체류시간을 고릅니다. B≥0.40, C≤0.25 mol/L와 비음수 출력을 요구합니다. 유량이 1 L/min이므로 체류시간을 줄이면 반응기 부피도 줄어듭니다. 구간 선형 surrogate를 만들고 그 ReLU 계산을 MILP 안에 표현합니다.

## Setup / 실행 준비

Extract the complete lab ZIP and run this notebook in that folder, with the scripts and supplied models beside it. Install requirements.txt into this notebook’s Python environment. The cells save new results under guided-results/.

실습 ZIP 전체를 풀고 스크립트·제공 모델과 같은 폴더에서 notebook을 실행합니다. Notebook의 Python 환경에 requirements.txt를 설치합니다. 새 결과는 guided-results/에 저장됩니다.

```text
python -m pip install -r requirements.txt
```

## 1. Follow scaling, hinges, and output contributions / 1. Scaling·hinge·출력 기여량 따라가기

Five reference knots at 2,4,6,8,10 min define linear interpolation. Scale tau by x=(tau−2)/8. Each hidden unit turns on after one threshold; its weight changes the output slope. At tau=5, only the first two hinges are active.

2,4,6,8,10분의 기준값 다섯 개 사이를 선형으로 잇습니다. x=(tau−2)/8로 scaling합니다. 각 은닉 뉴런은 문턱을 넘으면 켜지고 가중치만큼 출력 기울기를 바꿉니다. tau=5에서는 처음 두 hinge만 켜집니다.

In [1]:
from pathlib import Path
import numpy as np
from scipy.optimize import milp
import week05_lab as lab
net = lab.fit_network()
a,h,y = lab.forward(5., net)
contributions = h[:,None]*np.asarray(net["weights"])
print("scaled x:", (5-2)/8, "a:", a, "h:", h)
print("output bias:", net["bias"], "contributions by hidden unit:", contributions)
print("output:", y, "reference:", lab.reference(5.))
np.testing.assert_allclose(a, [.375,.125,-.125,-.375])
np.testing.assert_allclose(y, np.asarray(net["bias"])+contributions.sum(0))
np.testing.assert_allclose(y, (lab.reference(4.)+lab.reference(6.))/2)

scaled x: 0.375 a: [ 0.375  0.125 -0.125 -0.375] h: [0.375 0.125 0.    0.   ]
output bias: [0.8571428571428572, 0.28571428571428575, 0.057142857142857155] contributions by hidden unit: [[-0.28571429  0.14285714  0.14285714]
 [ 0.03463203 -0.03354978 -0.00108225]
 [ 0.         -0.         -0.        ]
 [ 0.         -0.         -0.        ]]
output: [0.60606061 0.39502165 0.19891775] reference: [0.6 0.4 0.2]


The surrogate at 5 min is the midpoint of the 4- and 6-min outputs, about [0.606061,0.395022,0.198918]. The physical reference is [0.6,0.4,0.2]. This interpolation error remains even when an MILP represents the neural network exactly.

5분 예측은 4분과 6분 출력의 중간값인 약 [0.606061,0.395022,0.198918]입니다. 공정 기준값은 [0.6,0.4,0.2]입니다. MILP가 신경망을 정확히 표현해도 이 보간 오차는 남습니다.

## 2. Read one matrix row as a constraint / 2. 행렬 한 행을 제약으로 읽기

Variables are [x,h0,h1,h2,h3,yA,yB,yC,z1,z2,z3]. The first row h0−x=0 encodes the always-active hinge. For threshold 0.25, preactivation a=x−0.25 lies in [−0.25,0.75]. Binary z selects inactive h=0 or active h=a through the two upper bounds and h≥a,h≥0.

변수 순서는 [x,h0,h1,h2,h3,yA,yB,yC,z1,z2,z3]입니다. 첫 행 h0−x=0은 항상 활성인 hinge를 나타냅니다. 문턱 0.25인 뉴런의 a=x−0.25는 [−0.25,0.75] 안에 있습니다. 이진 z가 두 상한과 h≥a,h≥0을 이용해 비활성 h=0 또는 활성 h=a를 선택합니다.

In [2]:
c, integrality, bounds, constraint = lab.formulate(net)
print("first row:", constraint.A[0], "row limits:", constraint.lb[0], constraint.ub[0])
print("binary positions:", np.flatnonzero(integrality))
fixed = lab.solve(net, fixed_tau=5., specifications=False)
v = fixed.x
print("fixed solution [x,h,y,z]:", v)
print("all row values:", constraint.A@v)
np.testing.assert_allclose(v[1:5], h, atol=1e-8)
np.testing.assert_allclose(v[5:8], y, atol=1e-8)

first row: [-1.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.] row limits: 0.0 0.0
binary positions: [ 8  9 10]
fixed solution [x,h,y,z]: [0.375      0.375      0.125      0.         0.         0.60606061
 0.39502165 0.19891775 1.         0.         0.        ]
all row values: [ 0.         -0.25        0.         -0.625      -0.375      -0.375
  0.         -0.375      -0.375       0.          0.85714286  0.28571429
  0.05714286]


Binary positions are 8,9,10, and the fixed hidden values are [0.375,0.125,0,0]. Disable product specifications during a fixed-input equivalence test: tau=5 fails the surrogate B≥0.40 specification, but still has a valid forward evaluation.

이진 변수 위치는 8,9,10이고 고정 hidden 값은 [0.375,0.125,0,0]입니다. 입력 고정 대조에서는 제품 규격을 끕니다. tau=5는 surrogate의 B≥0.40을 만족하지 않지만 forward 계산 자체는 유효합니다.

## 3. Optimize, restore the objective constant, and recheck / 3. 최적화 후 목적함수 상수를 복원하고 재검증하기

The solver minimizes 8x; adding the omitted constant 2 gives residence time. Recompute outputs using forward at the returned tau and then the process reference. The first comparison checks implementation; the second checks the selected process condition.

Solver는 8x를 최소화하므로 생략한 상수 2를 더해야 체류시간이 됩니다. 반환된 tau에서 forward를 다시 계산하고 공정 기준식도 계산합니다. 첫 비교는 구현을, 두 번째는 선택한 공정 조건을 검사합니다.

In [3]:
optimal = lab.solve(net)
tau = 2+8*optimal.x[0]
prediction = lab.forward(tau, net)[2]
truth = lab.reference(tau)
print("objective without constant:", optimal.fun, "tau:", tau)
print("surrogate:", prediction, "reference:", truth)
np.testing.assert_allclose(tau, 2+optimal.fun, atol=1e-8)
np.testing.assert_allclose(optimal.x[5:8], prediction, atol=1e-8)
np.testing.assert_allclose(tau, lab.enumerate_segments(net), atol=1e-8)
report = lab.run(Path("guided-results"))

objective without constant: 3.3538461538461615 tau: 5.3538461538461615
surrogate: [0.58461538 0.4        0.21538462] reference: [0.5794948  0.40413665 0.21636855]


The selected residence time is about 5.353846 min. B lies at the surrogate lower bound. Segment enumeration independently checks the one-dimensional optimum, and run also audits fixed inputs near activation boundaries. State whether the reference outputs meet every original specification.

선택한 체류시간은 약 5.353846분이며 surrogate B가 하한에 놓입니다. 구간별 계산이 1차원 최적해를 독립적으로 대조하고 run은 활성 경계 주변의 고정 입력도 검사합니다. 기준 출력이 원래 규격을 모두 만족하는지 적으세요.

## 4. Explain an infeasible result / 4. Infeasible 결과 설명하기

Raise only the B lower bound to 0.41 and keep C≤0.25. B peaks at about 0.410256 at 8 min, but C there exceeds its limit. C increases across all knots and reaches 0.25 between 6 and 8 min. Interpolate that crossing first; then read the largest B allowed before it.

B 하한만 0.41로 높이고 C≤0.25를 유지합니다. B는 8분에서 약 0.410256으로 최대이지만 그때 C가 상한을 넘습니다. C는 모든 knot에서 증가하고 6~8분 사이에 0.25가 됩니다. 먼저 이 교점을 보간한 뒤 그 이전에서 허용되는 최대 B를 읽습니다.

In [4]:
values = lab.forward(lab.KNOTS, net)[2]
tau_cap = 6+2*(.25-values[2,2])/(values[3,2]-values[2,2])
max_allowed_b = lab.forward(tau_cap, net)[2][1]
print("B and C at knots:", values[:,1:])
print("C-limited maximum tau:",tau_cap,"maximum allowed B:",max_allowed_b)
assert np.all(np.diff(values[:,2]) > 0) and np.all(np.diff(values[:4,1]) > 0)
c, integer, bounds, con = lab.formulate(net)
bounds.lb[6] = .41
infeasible = milp(c, integrality=integer, bounds=bounds, constraints=con)
print("status:", infeasible.status, infeasible.message)
assert max_allowed_b < .41 and infeasible.status == 2

B and C at knots: [[0.28571429 0.05714286]
 [0.38095238 0.15238095]
 [0.40909091 0.24545455]
 [0.41025641 0.32820513]
 [0.4        0.4       ]]
C-limited maximum tau: 6.109859154929577 maximum allowed B: 0.4091549295774648
status: 2 The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)


C restricts tau to at most 6.109859 min, where B is only 0.409155 mol/L. Thus both specifications cannot hold together for this frozen surrogate. Do not read result.x or report an operating point without a feasible solution. Submit the hand forward table, one decoded row, fixed-input comparison, selected point, reference check, and infeasibility argument.

C 때문에 tau는 최대 6.109859분이고 그때 B는 0.409155 mol/L입니다. 따라서 이 고정 surrogate에서는 두 규격을 함께 만족할 수 없습니다. 가능해가 없으면 result.x를 읽거나 운전점을 보고하지 않습니다. 손계산 표, 해석한 행 하나, 고정 입력 비교, 선택점과 기준 재검증, infeasible 근거를 제출하세요.

## Report / 보고서

Explain a hand prediction, the corresponding computed intermediate values, and the final interpretation. Change one setting and compare your predicted effect with the observed result. Keep training, validation/calibration, and held-out evaluation roles separate.

손으로 예측한 값, 대응하는 중간 계산값, 최종 해석을 연결해 설명합니다. 설정 하나를 바꾸고 예상한 효과와 실제 결과를 비교합니다. Train·validation/calibration·독립 평가 데이터의 역할을 구분합니다.